# ProtiCelli drug-perturbation profiling

Quantify drug-induced changes in subcellular protein distribution from
ProtiCelli-generated images.

| mode | inputs | reproduces |
|---|---|---|
| **validation** | real + generated | concordance vs ceiling (Fig. D), recovery AP (Fig. E), accuracy by effect decile (Fig. F), feature ranking (Table S4) |
| **profiling** | generated only | proteins ranked by simulated effect per drug, the antibody-free use |

Both modes share `profiling_core.py` and differ only in whether a real arm exists.
For real runs use `run_profiling.py`; this notebook is for inspection and small sets.

## Setup

In [ ]:
import os, glob, re
import numpy as np, pandas as pd
from tifffile import imread, imwrite
from tqdm.auto import tqdm
from profiling_core import extract_features, compute_effects, cliffs_delta, COVAR

## Configuration

In [ ]:
MODE       = "profiling"          # "validation" or "profiling"
IMAGE_DIR  = "example_images"
CONTROL    = "UNTREATED"
DRUGS      = ["PACLITAXEL", "VORINOSTAT"]
CELL_LINE  = "MDA-MB-468"
TOP_FEATURE = "rad0"              # radial distribution; ranked best on the reference set
# channel order (profiling_core): index 0=MT, 1=protein, 2=DAPI, 3=ER

## Generate images with ProtiCelli (profiling mode)

Skip if `*_pred.tif` already exist. ProtiCelli sees only the reference channels
plus the protein identity, never the drug, so recovered drug signal is inferred
from morphology. Output naming must match the parser:
`{prefix}_{TREAT}_..._crop_{n}__{CELL_LINE}_{GENE}_pred.tif`.

In [ ]:
# from proticelli import Model
# import torch
# model = Model()
# for ref_file in tqdm(sorted(glob.glob("reference_inputs/*.tif"))):
#     ref = imread(ref_file)
#     for gene in ["EIF5A", "MYH10", "VIM"]:            # your panel
#         res = model.predict(images=[ref], protein_names=[gene],
#                             cell_line_names=[CELL_LINE], num_inference_steps=50)
#         # imwrite(...) with a parser-matching name
# del model; torch.cuda.empty_cache()
print("illustrative; uncomment with proticelli installed")

## Demonstration: one image, one effect

Read a few features off a single image, and compute Cliff's delta between two toy
populations, so the effect size is transparent before the full run.

In [ ]:
demo = sorted(glob.glob(os.path.join(IMAGE_DIR, "*_pred.tif")))
if demo:
    f = extract_features(demo[0])
    for k in ["rad0", "rad2", "nuc_frac", "mean_cell"]:
        print(f"  {k:10s} {f[k]:.4f}")
rng = np.random.default_rng(0)
print("Cliff's delta demo:",
      round(cliffs_delta(rng.normal(.6,.2,200), rng.normal(.4,.2,200)), 3))

## Full run

For anything beyond a demo, use the command line (cached, resumable):

```bash
python run_profiling.py --mode validation --image_dir IMAGES \
    --control UNTREATED --drugs PACLITAXEL VORINOSTAT --out results
```

Validation writes `feature_ranking.csv` (Table S4), `recovery.csv` (Fig. D/E),
`decile.csv` (Fig. F). Profiling writes `profiling_{drug}.csv`, proteins ranked
by simulated effect.

In [ ]:
TAIL = re.compile(r"__(?P<line>[A-Za-z0-9\-]+)_(?P<gene>[A-Za-z0-9,\-]+)_(?P<kind>real|pred)\.tif$")
POS  = re.compile(r"_(?P<well>[A-Za-z0-9]+)_R\d+_z\d+_crop_\d+__")
TREATS = {CONTROL, *DRUGS}
def parse_name(path):
    fn = os.path.basename(path); t, p = TAIL.search(fn), POS.search(fn)
    if not (t and p): return None
    hits = {x.upper() for x in fn[:p.start()].split("_")} & TREATS
    if len(hits) != 1: return None
    kind = t.group("kind")
    return dict(kind=kind, gene=t.group("gene").split(",")[0],
                treat=hits.pop(), cell_id=fn[:-len(f"_{kind}.tif")])

want_real = MODE == "validation"
globs = ["*_pred.tif"] + (["*_real.tif"] if want_real else [])
paths = sorted(sum((glob.glob(os.path.join(IMAGE_DIR, g)) for g in globs), []))
rows = []
for pth in tqdm(paths):
    m = parse_name(pth); fe = extract_features(pth) if m else None
    if m and fe: rows.append({**m, **fe})
feat_df = pd.DataFrame(rows)
arms = ("real","pred") if want_real else ("pred",)
long, feats = compute_effects(feat_df, CONTROL, DRUGS, arms=arms)
print(len(long), "gene x drug effects,", "arms", arms)

## Profiling output

In [ ]:
if MODE == "profiling":
    for t in DRUGS:
        d = long[(long.arm=="pred") & (long.treat==t)][["gene", f"{TOP_FEATURE}|d"]]
        d = d.dropna().rename(columns={f"{TOP_FEATURE}|d":"sim_delta"})
        d = d.reindex(d.sim_delta.abs().sort_values(ascending=False).index)
        d["direction"] = np.where(d.sim_delta > 0, "periphery", "interior")  # rad0: shell 0 is OUTERMOST
        print(f"=== {t}: top 10 on {TOP_FEATURE} (rad0>0 = toward periphery) ==="); display(d.head(10).round(3))

## Caveat the method requires

The top feature (radial distribution) is computed on a mask from the reference
channels ProtiCelli conditions on, so strong recovery there can reflect cell
geometry rather than protein prediction. In profiling mode, treat rankings as
hypotheses and validate hits experimentally. Recovery is most reliable for large
effects; weak effects are near chance (Fig. F).